# NLBSE 2027 — E9-Fresh: Complete 15,000-Step CodeBERT Training

**Standalone, start-to-finish Kaggle notebook.** No E8 file, saved split, checkpoint, or previous notebook is required. It starts with the **official dataset** and freshly initialized `microsoft/codebert-base` weights. This is **not** a 10K → 15K continuation and must be reported separately.

**Experiment:** two-label multilabel classification · weighted BCE positive weights `[1, 10]` · 15,000 optimizer steps · batch size 8 · max input length 256 · peak LR 2e-5 · linear scheduler with 1,500 warmup steps · fixed prediction thresholds `[0.5,0.5]` · pilot evaluation every 1,000 steps. Select best checkpoint on **pilot only**.

**Set up Kaggle:** GPU Tesla T4 (or other CUDA GPU), Internet ON for Hugging Face dataset and model downloads. Run cells top-to-bottom. The expected historical count is about 193,072 training rows, 21,452 validation, pilot 5,000, clean holdout 16,420; counts are checked and printed, not used to force incompatible data. This run may not reproduce identical rows unless official dataset revision and splitting-library versions match earlier experiments.

**Important:** a fresh 15K experiment needs to process **120,000 examples** (15,000 × 8); the earlier E8 used 80,000. More steps does **not** guarantee improved hidden-test F1. Do not select a checkpoint by looking at the public test. The public split has been consulted historically; only an organizer hidden test is blind.

**Data privacy:** no uploaded tokens/passwords, no raw datasets committed to GitHub. Save Notebook Version or download backup to retain `/kaggle/working` outputs between Kaggle sessions.

## 1 — Environment and configuration
Install extra packages only if Kaggle imports fail. Do **not** install a CPU-only torch wheel over a working CUDA installation.

In [ ]:
# OPTIONAL (only if imports fail):
# %pip install -q 'transformers>=4.45,<6' 'datasets>=3,<5' 'scikit-learn>=1.4' safetensors
# Restart kernel after installation, then run all cells from the top.

In [ ]:
import os, gc, json, time, random, hashlib, platform, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, Sampler
from datasets import load_dataset
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

SEED = 42
DATASET_NAME = 'NLBSE/nlbse27-code-comment-classification'
MODEL_NAME = 'microsoft/codebert-base'
EXPERIMENT = 'E9_fresh_weighted_bce_scheduler_15000'
ROOT = Path('/kaggle/working/NLBSE2027')
OUT = ROOT / EXPERIMENT
BEST_DIR = OUT / 'best_model'
STATE_PATH = OUT / 'latest_training_state.pt'
OUT.mkdir(parents=True, exist_ok=True)

MAX_STEPS = 15_000
WARMUP_STEPS = 1_500
EVAL_EVERY = 1_000
MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
PILOT_SIZE = 5_000
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
POS_WEIGHTS = [1.0, 10.0]
THRESHOLDS = [0.5, 0.5]

assert torch.cuda.is_available(), 'Enable GPU in Kaggle Notebook Settings.'
DEVICE = torch.device('cuda')
def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_all(SEED)
print('Experiment:', EXPERIMENT)
print('Python:', platform.python_version(), 'Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Output:', OUT)

## 2 — Download official dataset and create fresh leakage-aware split
This cell **never searches for E8, `train.pkl`, or old notebooks**. It loads the official training split and removes exact-code hashes with conflicting labels. Then it applies seeded `StratifiedGroupKFold` (10 folds) and a stratified 5,000-row pilot subset. Exact pilot-code overlap is removed from the internal holdout. All projects may still overlap across train/validation.

In [ ]:
raw_train = load_dataset(DATASET_NAME, split='train')
original = raw_train.to_pandas().copy()
required = {'Function', 'Label', 'Projectname', 'Filepath'}
assert required.issubset(original.columns), f'Missing columns: {required - set(original.columns)}'

def hash_code(value):
    text = '' if value is None else str(value)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

original['code_hash'] = original['Function'].map(hash_code)
original['label_tuple'] = original['Label'].map(lambda y: tuple(int(v) for v in y))
assert original['label_tuple'].map(len).eq(2).all(), 'Expected two binary labels.'
assert all(v in (0,1) for pair in original['label_tuple'] for v in pair), 'Labels must be 0/1.'
conflict_counts = original.groupby('code_hash')['label_tuple'].nunique()
conflict_hashes = set(conflict_counts[conflict_counts > 1].index)
clean = original.loc[~original['code_hash'].isin(conflict_hashes)].reset_index(drop=True)
print('Original train:', len(original), 'Removed conflicting-label rows:', len(original)-len(clean))

splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
train_indices, val_indices = next(splitter.split(
    clean, clean['label_tuple'].astype(str).to_numpy(), clean['code_hash'].to_numpy()))
train_df = clean.iloc[train_indices].reset_index(drop=True)
val_df = clean.iloc[val_indices].reset_index(drop=True)
assert set(train_df['code_hash']).isdisjoint(set(val_df['code_hash'])), 'Hash leakage across train/validation!'

_, pilot_idx = train_test_split(
    np.arange(len(val_df)), test_size=PILOT_SIZE,
    random_state=SEED, stratify=val_df['label_tuple'].astype(str))
pilot_df = val_df.iloc[pilot_idx].reset_index(drop=True)
remaining_idx = np.setdiff1d(np.arange(len(val_df)), pilot_idx)
holdout_df = val_df.iloc[remaining_idx].reset_index(drop=True)
pilot_hashes = set(pilot_df['code_hash'])
clean_holdout_df = holdout_df.loc[
    ~holdout_df['code_hash'].isin(pilot_hashes)].reset_index(drop=True)
assert set(pilot_df['code_hash']).isdisjoint(set(clean_holdout_df['code_hash']))
assert len(train_df) >= MAX_STEPS * TRAIN_BATCH_SIZE, 'Insufficient train rows for a unique single-pass 15K run.'

np.save(OUT / 'pilot_val_indices.npy', pilot_idx)
print('Train:', len(train_df), '| Validation:', len(val_df))
print('Pilot:', len(pilot_df), '| Initial holdout:', len(holdout_df),
      '| Clean holdout:', len(clean_holdout_df))
print('Train positives:', np.asarray(train_df.Label.tolist(), dtype=int).sum(0).tolist())
print('Pilot positives:', np.asarray(pilot_df.Label.tolist(), dtype=int).sum(0).tolist())
print('Holdout positives:', np.asarray(clean_holdout_df.Label.tolist(), dtype=int).sum(0).tolist())
if (len(train_df), len(val_df), len(clean_holdout_df)) != (193072, 21452, 16420):
    print('NOTE: Row counts differ from historical E8; do not claim an identical split without checking hashes.')

## 3 — Reproducibility manifest
Record exact row-hash fingerprints and installed library versions. Identical sample counts alone do not prove identical data membership.

In [ ]:
import transformers, datasets, sklearn

def fingerprint(df):
    return hashlib.sha256('\n'.join(sorted(df['code_hash'].astype(str))).encode()).hexdigest()

fingerprints = {
    'train': fingerprint(train_df),
    'validation': fingerprint(val_df),
    'pilot': fingerprint(pilot_df),
    'clean_holdout': fingerprint(clean_holdout_df),
    'pilot_indices_sha256': hashlib.sha256(pilot_idx.tobytes()).hexdigest(),
}
manifest = {
    'experiment': EXPERIMENT, 'mode': 'fresh pretrained CodeBERT; NO E8 dependency',
    'model': MODEL_NAME, 'dataset': DATASET_NAME,
    'seed': SEED, 'max_steps': MAX_STEPS, 'warmup_steps': WARMUP_STEPS,
    'eval_every': EVAL_EVERY, 'max_length': MAX_LENGTH,
    'batch_size': TRAIN_BATCH_SIZE, 'eval_batch_size': EVAL_BATCH_SIZE,
    'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
    'positive_weights': POS_WEIGHTS, 'thresholds': THRESHOLDS,
    'train_size': len(train_df), 'validation_size': len(val_df),
    'pilot_size': len(pilot_df), 'clean_holdout_size': len(clean_holdout_df),
    'fingerprints': fingerprints,
    'versions': {'torch': torch.__version__, 'transformers': transformers.__version__,
                 'datasets': datasets.__version__, 'sklearn': sklearn.__version__,
                 'numpy': np.__version__, 'pandas': pd.__version__},
    'checkpoint_selection': 'maximum pilot mean F1 at fixed [0.5,0.5] thresholds',
    'notes': 'Single-seed, within-project split; public split not used for model selection.',
}
(OUT / 'run_config.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(fingerprints, indent=2))

## 4 — Dataset, tokenizer, and deterministic 120,000-example training order
Only one pass through 120,000 distinct randomly ordered training rows is used (no data reuse within this run).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class CodeDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame['Function'].fillna('').astype(str).tolist()
        self.labels = np.asarray(frame['Label'].tolist(), dtype=np.float32)
    def __len__(self): return len(self.texts)
    def __getitem__(self, index): return self.texts[index], self.labels[index]

def collate(batch):
    texts, ys = zip(*batch)
    x = tokenizer(list(texts), max_length=MAX_LENGTH, padding=True,
                  truncation=True, return_tensors='pt')
    x['labels'] = torch.tensor(np.stack(ys), dtype=torch.float32)
    return x

class FixedBatches(Sampler):
    def __init__(self, order, initial_step, final_step, batch_size):
        self.order = order; self.initial_step = initial_step
        self.final_step = final_step; self.batch_size = batch_size
    def __iter__(self):
        for s in range(self.initial_step, self.final_step):
            yield self.order[s*self.batch_size:(s+1)*self.batch_size]
    def __len__(self): return self.final_step - self.initial_step

train_data = CodeDataset(train_df)
pilot_data = CodeDataset(pilot_df)
holdout_data = CodeDataset(clean_holdout_df)
order = torch.randperm(len(train_data), generator=torch.Generator().manual_seed(SEED)).tolist()
assert len(order) >= MAX_STEPS * TRAIN_BATCH_SIZE
pilot_loader = DataLoader(pilot_data, batch_size=EVAL_BATCH_SIZE, collate_fn=collate, num_workers=0)
holdout_loader = DataLoader(holdout_data, batch_size=EVAL_BATCH_SIZE, collate_fn=collate, num_workers=0)
print('Distinct training samples to use:', MAX_STEPS*TRAIN_BATCH_SIZE)
print('Remaining train rows unused:', len(order)-MAX_STEPS*TRAIN_BATCH_SIZE)

## 5 — Initialize **fresh** CodeBERT, optimizer, and 15K scheduler
This notebook never loads an E8 checkpoint. It may resume only its **own E9-Fresh** saved state from the **same Kaggle session/output directory**, if one exists. To persist resume state across sessions, separately save/attach your E9 output.

In [ ]:
gc.collect(); torch.cuda.empty_cache()
seed_all(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, problem_type='multi_label_classification').to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=MAX_STEPS)
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(POS_WEIGHTS, device=DEVICE, dtype=torch.float32))

start_step, history, best_mean_f1, best_step, previous_elapsed = 0, [], -1.0, None, 0.0
if STATE_PATH.is_file():
    # Only load a training-state file you generated yourself. torch.load uses pickle.
    state = torch.load(STATE_PATH, map_location='cpu', weights_only=False)
    assert state['experiment'] == EXPERIMENT and state['fingerprints'] == fingerprints
    assert state['max_steps'] == MAX_STEPS
    model.load_state_dict(state['model'])
    optimizer.load_state_dict(state['optimizer'])
    scheduler.load_state_dict(state['scheduler'])
    start_step = int(state['step'])
    history = state['history']
    best_mean_f1, best_step = float(state['best_mean_f1']), state['best_step']
    previous_elapsed = float(state.get('elapsed_seconds', 0.0))
    random.setstate(state['rng_python'])
    np.random.set_state(state['rng_numpy'])
    torch.set_rng_state(state['rng_torch'])
    if 'rng_cuda' in state:
        torch.cuda.set_rng_state_all(state['rng_cuda'])
    assert len(history) and history[-1]['step'] == start_step
    assert BEST_DIR.is_dir(), 'Resume state found but best model folder is missing.'
    print('RESUMED OWN E9-FRESH STATE at step', start_step)
else:
    print('New model initialized from pretrained CodeBERT. No previous-run artifacts needed.')
print('Training remaining steps:', MAX_STEPS-start_step)

## 6 — Evaluation and recovery helper functions

In [ ]:
def evaluate(model, loader):
    model.eval()
    probabilities, actual = [], []
    with torch.inference_mode():
        for batch in loader:
            y = batch['labels'].numpy().astype(np.int64)
            x = {k: v.to(DEVICE) for k,v in batch.items() if k != 'labels'}
            p = torch.sigmoid(model(**x).logits).float().cpu().numpy()
            probabilities.append(p); actual.append(y)
    probabilities = np.concatenate(probabilities)
    actual = np.concatenate(actual)
    predicted = (probabilities >= np.asarray(THRESHOLDS)[None, :]).astype(int)
    result = {}
    all_f1 = []
    for j, name in enumerate(['Vulnerability', 'Technical Debt']):
        precision, recall, f1, _ = precision_recall_fscore_support(
            actual[:,j], predicted[:,j], average='binary', zero_division=0)
        result[name] = {
            'precision': float(precision), 'recall': float(recall), 'f1': float(f1),
            'confusion_matrix': confusion_matrix(actual[:,j], predicted[:,j], labels=[0,1]).tolist()}
        all_f1.append(float(f1))
    result['mean_f1'] = float(np.mean(all_f1))
    return result, probabilities, actual

def save_state(step, elapsed):
    state = {
        'experiment': EXPERIMENT, 'fingerprints': fingerprints,
        'max_steps': MAX_STEPS, 'step': step, 'history': history,
        'best_mean_f1': best_mean_f1, 'best_step': best_step,
        'elapsed_seconds': elapsed,
        'model': {k:v.detach().cpu() for k,v in model.state_dict().items()},
        'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
        'rng_python': random.getstate(), 'rng_numpy': np.random.get_state(),
        'rng_torch': torch.get_rng_state(),
        'rng_cuda': torch.cuda.get_rng_state_all(),
    }
    temp = STATE_PATH.with_suffix('.tmp')
    torch.save(state, temp)
    os.replace(temp, STATE_PATH)

print('Evaluation and state-saving helpers ready.')

## 7 — Train 15,000 steps (or resume the same E9 run)
This is the **only training cell**. It logs every 1,000 steps, saves the pilot-selected best model, and saves a restart state. No early stop. Avoid rerunning from step zero within the same live Python kernel.

In [ ]:
assert 0 <= start_step <= MAX_STEPS
session_start = time.time()
if start_step == MAX_STEPS:
    print('E9-Fresh training was already completed, skipping.')
else:
    batch_sampler = FixedBatches(order, start_step, MAX_STEPS, TRAIN_BATCH_SIZE)
    loader = DataLoader(train_data, batch_sampler=batch_sampler, collate_fn=collate, num_workers=0)
    running_loss, count = 0.0, 0
    for step, batch in enumerate(loader, start=start_step + 1):
        model.train()
        x = {k: v.to(DEVICE) for k, v in batch.items() if k != 'labels'}
        y = batch['labels'].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(**x).logits
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        running_loss += float(loss.item()); count += 1
        if step % EVAL_EVERY == 0:
            pilot_metrics, pilot_probs, pilot_truth = evaluate(model, pilot_loader)
            avg_loss = running_loss / max(count, 1)
            score = pilot_metrics['mean_f1']
            entry = {
                'step':step, 'learning_rate':float(scheduler.get_last_lr()[0]),
                'train_loss':avg_loss, **pilot_metrics
            }
            history.append(entry)
            print(f"Step {step}/{MAX_STEPS} | Loss={avg_loss:.4f} | "
                  f"LR={entry['learning_rate']:.2e} | "
                  f"Vuln F1={pilot_metrics['Vulnerability']['f1']:.4f} | "
                  f"Debt F1={pilot_metrics['Technical Debt']['f1']:.4f} | "
                  f"Mean F1={score:.4f}", flush=True)
            if score > best_mean_f1 + 1e-6:
                best_mean_f1 = score; best_step = step
                BEST_DIR.mkdir(parents=True, exist_ok=True)
                model.save_pretrained(BEST_DIR, safe_serialization=True)
                tokenizer.save_pretrained(BEST_DIR)
                np.savez_compressed(OUT / 'best_pilot_predictions.npz',
                                    probabilities=pilot_probs, labels=pilot_truth)
                (OUT / 'best_metrics.json').write_text(json.dumps({
                    'step': best_step, 'pilot_metrics': pilot_metrics,
                    'fixed_thresholds': THRESHOLDS}, indent=2))
                print('  New best model saved:', BEST_DIR, flush=True)
            (OUT / 'history.json').write_text(json.dumps(history, indent=2))
            save_state(step, previous_elapsed + time.time() - session_start)
            print('  Own E9 restart state saved at step:', step, flush=True)
            running_loss, count = 0.0, 0

print('Final step:', history[-1]['step'] if history else start_step)
print('Best step:', best_step, 'Best pilot Mean F1:', best_mean_f1)
print('Elapsed minutes:', round((previous_elapsed + time.time() - session_start)/60, 2))

## 8 — View training summary and plot F1
Do not mix the pilot scores with public-test metrics. The best checkpoint is pilot-selected; report this selection explicitly.

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
assert history, 'Complete at least one validation checkpoint.'
summary = pd.DataFrame([{
    'step': h['step'], 'learning_rate': h['learning_rate'],
    'train_loss': h['train_loss'],
    'vulnerability_f1': h['Vulnerability']['f1'],
    'debt_f1': h['Technical Debt']['f1'], 'mean_f1': h['mean_f1']
} for h in history])
display(summary)
fig, ax = plt.subplots(figsize=(9,4.5))
for col in ['vulnerability_f1', 'debt_f1', 'mean_f1']:
    ax.plot(summary.step, summary[col], marker='o', label=col)
ax.set_xlabel('Training step'); ax.set_ylabel('Pilot F1')
ax.set_title('E9-Fresh 15K — Validation history')
ax.legend(); ax.grid(alpha=.2); fig.tight_layout()
fig.savefig(OUT / 'pilot_f1_curve.png', dpi=170)
plt.show()
print('E8 historical best pilot F1: 0.7270 (separate training run)')
print('E9 current best pilot F1:', round(best_mean_f1, 5))

## 9 — Clean internal holdout evaluation (once, after model selection)
Do not use this holdout for checkpoint or threshold tuning. Exact pilot-code overlap is excluded, but all projects and many file identities remain shared with training. This is **not** unseen-project evaluation.

In [ ]:
assert history and history[-1]['step'] == MAX_STEPS, 'Finish the 15,000-step training first.'
assert BEST_DIR.is_dir(), 'Best checkpoint missing.'
best_model = AutoModelForSequenceClassification.from_pretrained(str(BEST_DIR)).to(DEVICE)
internal_metrics, _, _ = evaluate(best_model, holdout_loader)
internal_report = {
    'experiment': EXPERIMENT, 'best_step': best_step,
    'split': 'internal_holdout_excluding_exact_pilot_code_hashes',
    'rows': len(clean_holdout_df), 'max_length': MAX_LENGTH,
    'thresholds': THRESHOLDS,
    'limitations': 'within-project; many file IDs overlap train; not a blind test',
    'metrics': internal_metrics,
}
(OUT / 'clean_holdout_metrics.json').write_text(json.dumps(internal_report, indent=2))
print(json.dumps(internal_report, indent=2))
del best_model
gc.collect(); torch.cuda.empty_cache()

## 10 — Optional descriptive public evaluation (default OFF)
Only after freezing the checkpoint. Prior versions have already been assessed using the public split: this is **not** independent hidden-test generalization. Avoid additional model selection using this data.

In [ ]:
RUN_PUBLIC_EVALUATION = False  # optional; enable only after E9 is frozen
if RUN_PUBLIC_EVALUATION:
    assert history[-1]['step'] == MAX_STEPS
    public_df = load_dataset(DATASET_NAME, split='test').to_pandas()
    public_loader = DataLoader(CodeDataset(public_df), batch_size=EVAL_BATCH_SIZE,
                               collate_fn=collate, num_workers=0)
    frozen_model = AutoModelForSequenceClassification.from_pretrained(str(BEST_DIR)).to(DEVICE)
    t0 = time.perf_counter()
    public_metrics, _, _ = evaluate(frozen_model, public_loader)
    duration = time.perf_counter() - t0
    report = {'experiment': EXPERIMENT, 'split': 'historically consulted public test; NOT blind',
              'rows':len(public_df), 'best_step':best_step, 'max_length':MAX_LENGTH,
              'fixed_thresholds':THRESHOLDS, 'inference_seconds':duration,
              'metrics':public_metrics}
    (OUT/'public_descriptive_metrics.json').write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))
    del frozen_model; gc.collect(); torch.cuda.empty_cache()
else:
    print('Skipped public-test evaluation; checkpoint selection stays pilot-only.')

## 11 — Export research metadata ZIP + optional full checkpoint ZIP
The metadata ZIP is small. A FULL backup can be >1 GB because it includes both best model and optimizer training state. For local preservation download both the notebook `.ipynb` and full backup. Kaggle Save Version is recommended before leaving the session.

In [ ]:
METADATA_ZIP = Path('/kaggle/working/NLBSE2027_E9_FRESH_15K_METADATA.zip')
with zipfile.ZipFile(METADATA_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for name in ['run_config.json','pilot_val_indices.npy','history.json',
                 'best_metrics.json','clean_holdout_metrics.json',
                 'public_descriptive_metrics.json','pilot_f1_curve.png']:
        p = OUT / name
        if p.is_file(): zf.write(p, arcname=name)
print('Metadata ZIP:', METADATA_ZIP, 'size MB:', round(METADATA_ZIP.stat().st_size/2**20,2))

# Enable after training when you have sufficient free disk space (large ZIP):
MAKE_FULL_BACKUP = False
if MAKE_FULL_BACKUP:
    FULL_ZIP = Path('/kaggle/working/NLBSE2027_E9_FRESH_15K_FULL_BACKUP.zip')
    with zipfile.ZipFile(FULL_ZIP, 'w', compression=zipfile.ZIP_STORED,
                         allowZip64=True) as zf:
        for p in sorted(OUT.rglob('*')):
            if p.is_file():
                zf.write(p, arcname=str(p.relative_to(OUT.parent)))
    with zipfile.ZipFile(FULL_ZIP) as zf:
        assert zf.testzip() is None, 'Full backup ZIP integrity failed.'
    print('FULL ZIP:', FULL_ZIP, 'size GB:', round(FULL_ZIP.stat().st_size/2**30,2))

print('Best model:', BEST_DIR, 'exists:', BEST_DIR.is_dir())
print('Resume state:', STATE_PATH, 'exists:', STATE_PATH.is_file())

## 12 — What to report and what to do next

Send **dataset sizes/fingerprints**, **15 validation checkpoints**, **best pilot step and F1**, and **clean internal holdout** performance. E8 had historical pilot 0.7270, internal holdout 0.7703, public 0.7750; do not compare different splits without labels. Do not claim E9 is a continuation of E8. Be honest that model choice across the project has involved repeated public-split inspection.

**How to save Kaggle output locally:** after running the last cell, turn on `MAKE_FULL_BACKUP`, rerun ZIP cell, use Files/Output to download the ZIP, and separately download the notebook `.ipynb`. Keep the E8 backup safe as well.

**Stop rule:** don't launch a new experiment solely because E9 has a different result. Preserve the best checkpoint chosen by pilot validation, then prioritize paper, model-hosting, and competition submission. E9 is a fresh run, meaning 15K training from pretrained CodeBERT—not E8 10K + 5K.